# Risk Metrics

This notebook creates the consolidated Risk section CSV for the national tool. It combines socioeconomic risk and direct infrastructure-network risk while keeping hazard, scenario, and model run explicit in every row.

## 0. Setup

Country settings, administrative level, source paths, and model-run bundles come from `config/countries/KEN.toml`. Shared calculations live in `src/national_tool_metrics/sections/risk.py`.

In [ ]:
from pathlib import Path
import importlib
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import validate_section_output, write_section_output
import national_tool_metrics.sections.risk as risk_section

importlib.reload(risk_section)
from national_tool_metrics.sections.risk import (
    assemble_risk_run_metrics,
    build_capital_stock_risk_metrics,
    build_direct_network_risk_metrics,
    build_population_risk_metrics,
    combine_risk_run_outputs,
)

In [ ]:
config = load_country_config("KEN", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

river_run = config.risk_run("jrc_river_flood_baseline")
cyclone_run = config.risk_run("storm_tropical_cyclone_baseline_2020")

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
print(f"Risk runs: {river_run.name}, {cyclone_run.name}")

## 1. Socioeconomic Risk — Population

Load the JRC protection-adjusted expected annual flooded population (`AAR_protected`) and the `RP10`, `RP20`, `RP50`, `RP75`, `RP100`, `RP200`, and `RP500` event exposure metrics for the eight demographic groups and five national wealth quintiles. The risk map is encoded in each metric column name; unprotected AAR rows are intentionally excluded.

In [ ]:
population_risk_metrics = build_population_risk_metrics(
    config,
    admin_regions,
    river_run,
)
population_risk_metrics.head()

## 2. Socioeconomic Risk — Capital Stock

Load precomputed JRC protection-adjusted average annual losses and `RP10`, `RP20`, `RP50`, `RP75`, `RP100`, `RP200`, and `RP500` event losses for residential, non-residential, infrastructure, and total capital stock. The risk map is encoded in each metric column name.

In [ ]:
capital_stock_risk_metrics = build_capital_stock_risk_metrics(
    config,
    admin_regions,
    river_run,
)
capital_stock_risk_metrics.head()

## 3. Direct Infrastructure Risk — Roads and Rail

Allocate JRC river-flood expected annual damage to administrative regions by intersected network length. Road EAD is reported as a total and by road class; rail is reported as a total.

In [ ]:
river_direct_risk_metrics = build_direct_network_risk_metrics(
    config,
    admin_regions,
    river_run,
)
river_direct_risk_metrics.head()

## 4. Direct Infrastructure Risk — Power

Allocate STORM tropical-cyclone power-network EAD to administrative regions. An all-zero result is retained as valid for Kenya; the same workflow can be reused in countries with greater tropical-cyclone exposure.

In [ ]:
cyclone_direct_risk_metrics = build_direct_network_risk_metrics(
    config,
    admin_regions,
    cyclone_run,
)
cyclone_direct_risk_metrics.head()

## 5. Deferred Risk Components

**Indirect infrastructure risk** will be added for tropical cyclone only when its input schema is available. **Social-infrastructure risk** is produced by a separate colleague workflow and is not calculated here. Both can later be added as metric tables without changing the canonical Risk output grain.

## 6. Combine and Validate

Create one row per administrative region, hazard, scenario, and model run. Blank cells mean that a metric does not apply to that run; numeric zero means the applicable calculation returned zero risk.

In [ ]:
river_risk_metrics = assemble_risk_run_metrics(
    config,
    admin_regions,
    river_run,
    [
        population_risk_metrics,
        capital_stock_risk_metrics,
        river_direct_risk_metrics,
    ],
)
cyclone_risk_metrics = assemble_risk_run_metrics(
    config,
    admin_regions,
    cyclone_run,
    [cyclone_direct_risk_metrics],
)

risk_metrics = combine_risk_run_outputs(
    [river_risk_metrics, cyclone_risk_metrics]
)
validate_section_output(risk_metrics, "risk")

print(f"Rows: {len(risk_metrics):,}")
print(f"Metric columns: {len(risk_metrics.columns) - 9:,}")
risk_metrics.groupby(["hazard", "scenario", "model_run"]).size()

## 7. Export

Write the canonical Risk CSV after reviewing the combined table and run counts above.

In [ ]:
output_path = write_section_output(risk_metrics, config, "risk")
print(f"Exported Risk metrics to: {output_path}")